# Практическая Работа - Основы Рекомендательных Систем

**Тема:** Введение в Рекомендательные Системы (РС)

**Цель работы:** Практическое освоение двух базовых подходов к построению РС: контентной фильтрации (Content-Based) и коллаборативной фильтрации (User-Based CF).

## 0. Подготовка Рабочей Среды

Для выполнения работы вам потребуется Python (рекомендуется 3.10+) и следующие библиотеки:

- **pandas** — для работы с данными (таблицами)
- **scikit-learn (sklearn)** — для расчета косинусного сходства
- **scikit-surprise (surprise)** — специализированная библиотека для РС, которая упрощает работу с коллаборативной фильтрацией

### 0.1. Установка Библиотек

Откройте терминал и выполните команду:


In [1]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, KNNWithMeans, accuracy
from surprise.model_selection import train_test_split

print("Библиотеки успешно загружены!")

Библиотеки успешно загружены!


---

## Задание 1. Контентная Фильтрация (Content-Based Filtering)

**Цель:** Построить простейшую систему, которая рекомендует фильмы на основе их жанрового сходства с фильмом, который пользователь оценил.

### 1.1. Исходные Данные (Моделирование)

Представьте, что у нас есть 5 фильмов, описанных по 4 жанрам (1 - есть жанр, 0 - нет жанра):

| Film_ID | Title | Action | Comedy | Drama | Sci-Fi |
|---------|-------|--------|--------|-------|--------|
| 1 | Die Hard | 1 | 0 | 0 | 0 |
| 2 | The Martian | 0 | 0 | 0 | 1 |
| 3 | Pulp Fiction | 0 | 0 | 1 | 0 |
| 4 | Guardians of the Galaxy | 1 | 1 | 0 | 1 |
| 5 | La La Land | 0 | 0 | 1 | 0 |

### 1.2. Шаги Выполнения

#### Шаг 1: Создание Матрицы Признаков

Создайте DataFrame `movies_df` и извлеките из него только жанровые столбцы, чтобы получить матрицу признаков `M`.

In [2]:
# Шаг 1: Создание матрицы признаков

# Создание DataFrame с данными о фильмах
data = {
    'Film_ID': [1, 2, 3, 4, 5],
    'Title': ['Die Hard', 'The Martian', 'Pulp Fiction', 'Guardians of the Galaxy', 'La La Land'],
    'Action': [1, 0, 0, 1, 0],
    'Comedy': [0, 0, 0, 1, 0],
    'Drama': [0, 0, 1, 0, 1],
    'Sci-Fi': [0, 1, 0, 1, 0]
}

movies_df = pd.DataFrame(data).set_index('Film_ID')
print("Данные о фильмах:")
print(movies_df)

# Извлечение матрицы признаков (только жанры)
feature_matrix = movies_df[['Action', 'Comedy', 'Drama', 'Sci-Fi']]
print("\nМатрица Признаков:")
print(feature_matrix)

Данные о фильмах:
                           Title  Action  Comedy  Drama  Sci-Fi
Film_ID                                                        
1                       Die Hard       1       0      0       0
2                    The Martian       0       0      0       1
3                   Pulp Fiction       0       0      1       0
4        Guardians of the Galaxy       1       1      0       1
5                     La La Land       0       0      1       0

Матрица Признаков:
         Action  Comedy  Drama  Sci-Fi
Film_ID                               
1             1       0      0       0
2             0       0      0       1
3             0       0      1       0
4             1       1      0       1
5             0       0      1       0


#### Шаг 2: Расчет Косинусного Сходства

Используйте функцию `cosine_similarity` из `sklearn.metrics.pairwise` для расчета матрицы сходства между всеми фильмами.

In [3]:
# Шаг 2: Расчет матрицы сходства

# Каждая ячейка [i, j] будет содержать сходство между фильмом i и фильмом j
cosine_sim_matrix = cosine_similarity(feature_matrix)

# Создание DataFrame для удобного просмотра
cosine_sim_df = pd.DataFrame(
    cosine_sim_matrix,
    index=movies_df['Title'],
    columns=movies_df['Title']
)

print("\nМатрица Косинусного Сходства:")
print(cosine_sim_df.round(3))


Матрица Косинусного Сходства:
Title                    Die Hard  The Martian  Pulp Fiction  \
Title                                                          
Die Hard                    1.000        0.000           0.0   
The Martian                 0.000        1.000           0.0   
Pulp Fiction                0.000        0.000           1.0   
Guardians of the Galaxy     0.577        0.577           0.0   
La La Land                  0.000        0.000           1.0   

Title                    Guardians of the Galaxy  La La Land  
Title                                                         
Die Hard                                   0.577         0.0  
The Martian                                0.577         0.0  
Pulp Fiction                               0.000         1.0  
Guardians of the Galaxy                    1.000         0.0  
La La Land                                 0.000         1.0  


#### Шаг 3: Функция Рекомендации

Напишите функцию `get_recommendations(title, cosine_sim_df, movies_df, top_n=2)`, которая:
1. Находит строку сходства для заданного фильма (`title`)
2. Сортирует фильмы по убыванию сходства
3. Возвращает `top_n` самых похожих фильмов (исключая сам исходный фильм)

In [4]:
# Шаг 3: Функция рекомендации

def get_recommendations(title, cosine_sim_df, movies_df, top_n=2):
    """
    Рекомендует фильмы на основе косинусного сходства.
    
    Parameters:
    -----------
    title : str
        Название фильма, для которого ищем рекомендации
    cosine_sim_df : pd.DataFrame
        Матрица косинусного сходства
    movies_df : pd.DataFrame
        DataFrame с данными о фильмах
    top_n : int
        Количество рекомендуемых фильмов
    
    Returns:
    --------
    list : Список названий рекомендуемых фильмов
    """
    # Получаем оценки сходства для заданного фильма
    sim_scores = cosine_sim_df[title]
    
    # Сортируем фильмы по убыванию сходства
    sim_scores = sim_scores.sort_values(ascending=False)
    
    # Исключаем сам фильм и берем top_n
    # [1:top_n+1] - пропускаем первый элемент (сам фильм)
    top_recommendations = sim_scores.iloc[1:top_n+1]
    
    # Возвращаем названия фильмов
    return list(top_recommendations.index)

# Пример использования:
# Если пользователь посмотрел "Die Hard" (Боевик), что ему порекомендует система?
recommendations_die_hard = get_recommendations('Die Hard', cosine_sim_df, movies_df, top_n=2)
print(f"\nРекомендации для 'Die Hard': {recommendations_die_hard}")

# Дополнительные примеры
print(f"Рекомендации для 'The Martian': {get_recommendations('The Martian', cosine_sim_df, movies_df, top_n=2)}")
print(f"Рекомендации для 'Pulp Fiction': {get_recommendations('Pulp Fiction', cosine_sim_df, movies_df, top_n=2)}")
print(f"Рекомендации для 'Guardians of the Galaxy': {get_recommendations('Guardians of the Galaxy', cosine_sim_df, movies_df, top_n=2)}")
print(f"Рекомендации для 'La La Land': {get_recommendations('La La Land', cosine_sim_df, movies_df, top_n=2)}")


Рекомендации для 'Die Hard': ['Guardians of the Galaxy', 'The Martian']
Рекомендации для 'The Martian': ['Guardians of the Galaxy', 'Die Hard']
Рекомендации для 'Pulp Fiction': ['La La Land', 'Die Hard']
Рекомендации для 'Guardians of the Galaxy': ['Die Hard', 'The Martian']
Рекомендации для 'La La Land': ['La La Land', 'Die Hard']


### 1.3. Вопросы для Отчета

**1. Какое сходство (число) система рассчитала между фильмами "The Martian" и "Guardians of the Galaxy"? Объясните, почему.**

**2. Какой главный недостаток Content-Based фильтрации вы видите на примере фильма "La La Land"? (Подсказка: посмотрите, что ему рекомендуют).**

**Ваши ответы:**

1. Сходство между "The Martian" и "Guardians of the Galaxy":

2. Главный недостаток Content-Based фильтрации на примере "La La Land":

---

## Задание 2. Коллаборативная Фильтрация (User-Based CF)

**Цель:** Использовать библиотеку `surprise` для реализации User-Based CF с метрикой Пирсона и оценить ее качество.

### 2.1. Исходные Данные

Мы будем использовать встроенный набор данных **MovieLens 100k**, который содержит 100,000 оценок от 943 пользователей для 1682 фильмов.

### 2.2. Шаги Выполнения

#### Шаг 1: Загрузка и Подготовка Данных

Загрузите данные и разделите их на обучающую (`trainset`) и тестовую (`testset`) выборки.

In [5]:
# Шаг 1: Загрузка и подготовка данных

# Загрузка данных MovieLens 100k
# Reader нужен, чтобы указать диапазон оценок (от 1 до 5)
data = Dataset.load_builtin('ml-100k')

# Разделение данных на обучающую и тестовую выборки (80% на 20%)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {trainset.n_ratings} оценок")
print(f"Размер тестовой выборки: {len(testset)} оценок")

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to C:\Users\melni/.surprise_data/ml-100k
Размер обучающей выборки: 80000 оценок
Размер тестовой выборки: 20000 оценок


#### Шаг 2: Обучение Модели User-Based CF

Используйте алгоритм `KNNWithMeans` (K-Nearest Neighbors с учетом средних оценок), который реализует User-Based CF с корреляцией Пирсона.

- `k=40`: Количество соседей, которых мы ищем
- `sim_options`: Настройки сходства
  - `name='pearson'` — использование корреляции Пирсона
  - `user_based=True` — User-Based CF

In [6]:
# Шаг 2: Настройка и обучение модели User-Based CF

# Используем корреляцию Пирсона (pearson) и User-Based подход
sim_options = {
    'name': 'pearson',
    'user_based': True  # User-Based Collaborative Filtering
}

# Инициализация алгоритма
algo = KNNWithMeans(k=40, sim_options=sim_options)

# Обучение модели
algo.fit(trainset)
print("\nМодель User-Based CF (Пирсон) обучена.")

Computing the pearson similarity matrix...
Done computing similarity matrix.

Модель User-Based CF (Пирсон) обучена.


#### Шаг 3: Оценка Качества Модели

Проверьте качество предсказаний на тестовой выборке, используя метрику **RMSE** (Root Mean Square Error).

In [7]:
# Шаг 3: Оценка качества модели

# Предсказание на тестовой выборке
predictions = algo.test(testset)

# Расчет RMSE
rmse = accuracy.rmse(predictions, verbose=True)
print(f"\nRMSE (Среднеквадратичная ошибка) на тестовой выборке: {rmse:.4f}")

RMSE: 0.9492

RMSE (Среднеквадратичная ошибка) на тестовой выборке: 0.9492


#### Шаг 4: Предсказание Оценки для Конкретного Пользователя

Предскажите, какую оценку поставит конкретный пользователь (например, `user_id=196`) конкретному фильму (например, `item_id=302` — фильм "L.A. Confidential").

In [8]:
# Шаг 4: Предсказание для конкретной пары (Пользователь, Объект)

user_id = '196'
item_id = '302'  # Фильм "L.A. Confidential"

# Получение предсказания
prediction = algo.predict(user_id, item_id, verbose=True)
print(f"\nПредсказанная оценка для Пользователя {user_id} (Фильм {item_id}): {prediction.est:.3f}")

user: 196        item: 302        r_ui = None   est = 4.18   {'actual_k': 40, 'was_impossible': False}

Предсказанная оценка для Пользователя 196 (Фильм 302): 4.175


#### Шаг 5: Сравнение с Косинусным Сходством (Бонус)

Проверьте, как изменится RMSE при использовании косинусного сходства вместо корреляции Пирсона.

In [9]:
# Шаг 5: Сравнение с косинусным сходством

# Настройка с косинусным сходством
sim_options_cosine = {
    'name': 'cosine',
    'user_based': True
}

# Инициализация и обучение модели
algo_cosine = KNNWithMeans(k=40, sim_options=sim_options_cosine)
algo_cosine.fit(trainset)

# Оценка качества
predictions_cosine = algo_cosine.test(testset)
rmse_cosine = accuracy.rmse(predictions_cosine, verbose=True)

print(f"\nRMSE с косинусным сходством: {rmse_cosine:.4f}")
print(f"RMSE с корреляцией Пирсона: {rmse:.4f}")
print(f"Разница: {rmse - rmse_cosine:.4f}")

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9538

RMSE с косинусным сходством: 0.9538
RMSE с корреляцией Пирсона: 0.9492
Разница: -0.0047


### 2.3. Вопросы для Отчета

**1. Что означает полученное вами значение RMSE? Если бы RMSE было равно 0.5, а не полученному вами значению, что бы это значило для точности предсказаний?**

**2. Как изменится значение RMSE, если в настройках `sim_options` заменить `'pearson'` на `'cosine'` (Косинусное сходство)? Объясните, почему Пирсон часто дает лучший результат, чем Косинусное сходство, в контексте оценок пользователей.**

**Ваши ответы:**

1. Значение RMSE и его интерпретация:

2. Сравнение Пирсона и Косинусного сходства:

---

## Требования к Оформлению Отчета

Отчет должен быть оформлен в виде Markdown-файла или Jupyter Notebook и содержать:

1. Полный код для Задания 1 и Задания 2
2. Результаты выполнения кода (вывод в консоль)
3. Ответы на все вопросы в разделах 1.3 и 2.3
4. Краткий вывод о том, какой из двух подходов (Content-Based или User-Based CF) кажется вам более перспективным для крупного онлайн-кинотеатра, и почему

## Выводы

*(Напишите ваш вывод здесь)*

Какой подход (Content-Based или User-Based CF) кажется вам более перспективным для крупного онлайн-кинотеатра и почему?